# PIPELINE PREPROCESSING

In [1]:
superprompt = """
Descrivi questa immagine in circa 3 frasi in modo diretto e concreto, concentrandoti su elementi visibili, azioni, funzioni o informazioni presenti. Evita frasi generiche come 'l’immagine mostra', 'si vede', 'c’è', ecc. Usa frasi informative che possano essere immediatamente utili per un sistema di recupero basato su contenuti (RAG).
"""

In [2]:
import json
import logging
import time
from pathlib import Path
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import ImageRefMode, PictureItem
from docling.datamodel.pipeline_options import granite_picture_description, RapidOcrOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    PictureDescriptionVlmOptions,
)
import os


_log = logging.getLogger(__name__)
conv_result = []

cartellainput = "procedure"

logging.basicConfig(level=logging.INFO)

    

    # Configurazione pipeline con descrizione immagini
pipeline_options = PdfPipelineOptions()
pipeline_options.do_table_structure = True
pipeline_options.do_ocr = True
pipeline_options.ocr_options = RapidOcrOptions(lang=["ita", "eng"])
#    pipeline_options.generate_picture_images=True #CONTROLLARE
pipeline_options.do_picture_description = True
pipeline_options.picture_description_options=PictureDescriptionVlmOptions(
    repo_id="Qwen/Qwen3-VL-8B-Instruct-FP8",
        # prompt="descrivi l'immagine in tre frasi. Sii conciso e accurato.",
    prompt = superprompt
)
    # pipeline_options.picture_description_options.prompt = (
    # "Describe the image in three sentences. Be consise and accurate."
    # )

pipeline_options.table_structure_options.do_cell_matching = True
    # pipeline_options.ocr_options.lang = ["ita", "eng"]
pipeline_options.images_scale = 2.0
pipeline_options.accelerator_options = AcceleratorOptions(
        num_threads=6, device=AcceleratorDevice.AUTO
)

   

doc_converter = DocumentConverter(
    format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)



#per ogni file nella cartella di input
for file in os.listdir(cartellainput):

    start_time = time.time()
    conv_result = doc_converter.convert(cartellainput + "/" + file) #riga pesante
    end_time = time.time() - start_time

    print(f"Conversione completata di {file} in {end_time:.2f} secondi.")

    #Logica generazione output
    doc_filename = conv_result.input.file.stem
    output_dir = Path("scratch") / doc_filename
    output_dir.mkdir(parents=True, exist_ok=True)

    #sposto il file preprocessato da preprocessing a procedure_preprocessate
    os.replace(cartellainput + "/" + file, "procedure_preprocessate/"+doc_filename+".pdf")



    # Export Docling document JSON format (include le descrizioni):
    with (output_dir / f"{doc_filename}.json").open("w", encoding="utf-8") as fp:
        fp.write(json.dumps(conv_result.document.export_to_dict()))

    # Export Markdown format (le descrizioni saranno incluse):
    with (output_dir / f"{doc_filename}.md").open("w", encoding="utf-8") as fp:
        fp.write(conv_result.document.export_to_markdown())

    # Export Text format:
    # with (output_dir / f"{doc_filename}.txt").open("w", encoding="utf-8") as fp:
    #     fp.write(conv_result.document.export_to_markdown())

    # Export Document Tags format:
    # with (output_dir / f"{doc_filename}.doctags").open("w", encoding="utf-8") as fp:
    #     fp.write(conv_result.document.export_to_doctags())


2026-01-16 17:02:12,098 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-16 17:02:12,115 - INFO - Going to convert document batch...
2026-01-16 17:02:12,116 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 18aa5f175f087cd88bd47e1ae3c12fbe
2026-01-16 17:02:12,119 - INFO - Loading plugin 'docling_defaults'
2026-01-16 17:02:12,120 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-01-16 17:02:12,416 - INFO - Accelerator device: 'cuda:0'


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-01-16 17:02:20,214 - INFO - Loading plugin 'docling_defaults'
2026-01-16 17:02:20,217 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-01-16 17:02:20,331 - INFO - Accelerator device: 'cuda:0'
[INFO] 2026-01-16 17:02:20,338 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-01-16 17:02:20,363 [RapidOCR] download_file.py:60: File exists and is valid: /home/dev/workspace/preprocessing/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-01-16 17:02:20,363 [RapidOCR] main.py:53: Using /home/dev/workspace/preprocessing/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-01-16 17:02:20,400 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-01-16 17:02:20,401 [RapidOCR] download_file.py:60: File exists and is valid: /home/dev/workspace/preprocessing/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infe

Conversione completata di UTL_CR_WAMAS_tabella.pdf in 13.52 secondi.


# VLM Pipeline (silent error... )

In [ ]:
import json
import logging
import time
from pathlib import Path
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
import os
from docling.pipeline.vlm_pipeline import VlmPipeline, VlmPipelineOptions



cartellainput = "procedure"


pipeline_options = VlmPipelineOptions()
pipeline_options.vlm_options.repo_id = "Qwen/Qwen3-VL-8B-Instruct-FP8"


pipeline_options.accelerator_options = AcceleratorOptions(
    num_threads=6, 
    device=AcceleratorDevice.AUTO
)


doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=VlmPipeline,  
            pipeline_options=pipeline_options
        )
    }
)


for file in os.listdir(cartellainput):

    start_time = time.time()
    conv_result = doc_converter.convert(cartellainput + "/" + file) #riga pesante
    end_time = time.time() - start_time

    print(f"Conversione completata di {file} in {end_time:.2f} secondi.")

    #Logica generazione output
    doc_filename = conv_result.input.file.stem
    output_dir = Path("scratch") / doc_filename
    output_dir.mkdir(parents=True, exist_ok=True)

    #sposto il file preprocessato da preprocessing a procedure_preprocessate
    os.replace(cartellainput + "/" + file, "procedure_preprocessate/"+doc_filename+".pdf")



    # Export Docling document JSON format (include le descrizioni):
    with (output_dir / f"{doc_filename}.json").open("w", encoding="utf-8") as fp:
        fp.write(json.dumps(conv_result.document.export_to_dict()))

    # Export Markdown format (le descrizioni saranno incluse):
    with (output_dir / f"{doc_filename}.md").open("w", encoding="utf-8") as fp:
        fp.write(conv_result.document.export_to_markdown())

    # Export Text format:
    # with (output_dir / f"{doc_filename}.txt").open("w", encoding="utf-8") as fp:
    #     fp.write(conv_result.document.export_to_markdown())

    # Export Document Tags format:
    # with (output_dir / f"{doc_filename}.doctags").open("w", encoding="utf-8") as fp:
    #     fp.write(conv_result.document.export_to_doctags())


2026-01-13 14:49:30,230 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-13 14:49:30,232 - INFO - Going to convert document batch...
2026-01-13 14:49:30,232 - INFO - Initializing pipeline for VlmPipeline with options hash 5e06f930eda010178dd07dd75a72ee08
2026-01-13 14:49:30,233 - INFO - Accelerator device: 'cuda:0'


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.16 GiB. GPU 0 has a total capacity of 31.35 GiB of which 248.88 MiB is free. Including non-PyTorch memory, this process has 26.98 GiB memory in use. Of the allocated memory 25.54 GiB is allocated by PyTorch, and 877.44 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)